# Member 1: Sepsis-3 Cohort Extraction, Preprocessing, and State-Action Encoding

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import joblib

sys.path.append('..')
from member_1_data.synthetic_data import generate_sepsis_cohort
from member_1_data.preprocessor import ClinicalPreprocessor
from member_1_data.state_action_encoder import StateActionEncoder
from member_1_data.data_quality import run_data_quality_audit

os.makedirs('../data', exist_ok=True)
os.makedirs('../artifacts', exist_ok=True)
os.makedirs('../docs', exist_ok=True)


### 1. Cohort Generation and Data Quality Audit

In [ ]:
raw_df = generate_sepsis_cohort(n_patients=600, random_state=42)
raw_csv_path = '../data/raw_sepsis_cohort.csv'
raw_df.to_csv(raw_csv_path, index=False)

print("Total observations:", len(raw_df))
print("Unique patients:", raw_df["subject_id"].nunique())
print("Crude mortality rate:", f"{raw_df.groupby('subject_id')['died'].first().mean()*100:.2f}%")

audit_summary = run_data_quality_audit(raw_csv_path, '../docs/data_quality_report.md')
raw_df.head(6)


### 2. Patient-Level Splitting (Strict Zero Leakage)

In [ ]:
preprocessor = ClinicalPreprocessor()
train_raw, val_raw, test_raw = preprocessor.split_patients(raw_df, train_ratio=0.70, val_ratio=0.15, random_state=42)

train_subjs = set(train_raw["subject_id"])
val_subjs = set(val_raw["subject_id"])
test_subjs = set(test_raw["subject_id"])

print("Train patients:", len(train_subjs))
print("Val patients:  ", len(val_subjs))
print("Test patients: ", len(test_subjs))
assert len(train_subjs.intersection(val_subjs)) == 0
assert len(train_subjs.intersection(test_subjs)) == 0
assert len(val_subjs.intersection(test_subjs)) == 0
print("Zero patient leakage verified across all splits.")


### 3. Feature Imputation and Standardization (Fitted Exclusively on Train)

In [ ]:
train_scaled = preprocessor.fit_transform(train_raw)
val_scaled = preprocessor.transform(val_raw)
test_scaled = preprocessor.transform(test_raw)

joblib.dump(preprocessor, '../artifacts/preprocessor.joblib')
print("Fitted preprocessor saved to artifacts/preprocessor.joblib.")
train_scaled[[c + '_scaled' for c in preprocessor.feature_cols[:6]]].describe()


### 4. K-means++ State Clustering and 5x5 Action Discretization

In [ ]:
scaled_feature_cols = [c + '_scaled' for c in preprocessor.feature_cols]
encoder = StateActionEncoder(n_states=75, random_state=42)
encoder.fit(train_scaled, scaled_feature_cols)

encoder.export_action_bins('../artifacts/action_bins.json')
encoder.export_manifest(
    '../artifacts/cohort_manifest.json',
    n_patients_total=raw_df['subject_id'].nunique(),
    n_train=train_raw['subject_id'].nunique(),
    n_val=val_raw['subject_id'].nunique(),
    n_test=test_raw['subject_id'].nunique()
)
encoder.export_features_schema('../artifacts/features_schema.json')
joblib.dump(encoder, '../artifacts/state_action_encoder.joblib')

print("Health states:", encoder.n_states)
print("Alive absorbing state:", encoder.alive_state)
print("Dead absorbing state:", encoder.dead_state)
print("Fluid dose cutoffs (mL/4h):", encoder.fluid_bins)
print("Vasopressor dose cutoffs (mcg/kg/min):", encoder.vaso_bins)


### 5. Sequential Trajectory Construction (Data Contract Verification)

In [ ]:
tr_train = encoder.build_trajectories(train_scaled, split_name='train')
tr_val = encoder.build_trajectories(val_scaled, split_name='val')
tr_test = encoder.build_trajectories(test_scaled, split_name='test')

tr_train.to_csv('../data/trajectories_train.csv', index=False)
tr_val.to_csv('../data/trajectories_val.csv', index=False)
tr_test.to_csv('../data/trajectories_test.csv', index=False)

print("Train trajectory steps:", len(tr_train))
print("Val trajectory steps:  ", len(tr_val))
print("Test trajectory steps: ", len(tr_test))
print("Data contract columns:", list(tr_train.columns))
tr_train.head(8)
